# MS-GARCH: Mudanca de Regime na Volatilidade

Neste notebook, exploramos o modelo **MS-GARCH** (Markov-Switching GARCH), que combina
a modelagem de **volatilidade condicional** (GARCH) com **mudanca de regime** (Markov-Switching).

A ideia central e que os mercados financeiros alternam entre periodos de **calma**
(volatilidade baixa) e **turbulencia** (volatilidade alta), e que essa transicao
segue uma dinamica de Markov.

**Conteudo:**
1. Por que MS-GARCH?
2. MS(2)-GARCH(1,1): estimacao
3. Regime de alta vs baixa volatilidade
4. Volatilidade condicional com regime-switching
5. MS-GARCH vs GARCH padrao

**Referencias:**
- Haas, M., Mittnik, S. & Paolella, M.S. (2004). *A New Approach to Markov-Switching GARCH Models*. Journal of Financial Econometrics, 2(4), 493-530.
- Gray, S.F. (1996). *Modeling the Conditional Distribution of Interest Rates as a Regime-Switching Process*. Journal of Financial Economics, 42, 27-62.
- Hamilton, J.D. (1989). *A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle*. Econometrica.
- Krolzig, H.-M. (1997). *Markov-Switching Vector Autoregressions*. Springer.

In [ ]:
import sys

sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from utils.plot_helpers import (
    plot_ms_volatility,
    plot_regime_probabilities,
    plot_transition_matrix,
)

from archbox.models import GARCH
from archbox.regime import (
    MarkovSwitchingGARCH,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

# Carregar dados de retornos do S&P 500
data = pd.read_csv('../data/sp500_returns.csv', parse_dates=['date'], index_col='date')
returns = data['returns']

print(f"Periodo: {returns.index[0].date()} a {returns.index[-1].date()}")
print(f"Observacoes: {len(returns)} (diarias)")
print("\nEstatisticas descritivas:")
print(data[['returns']].describe())

## 1. Por que MS-GARCH?

O modelo **GARCH(1,1)** padrao assume que os parametros $\omega, \alpha, \beta$ sao **constantes**
ao longo de toda a amostra:

$$\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2$$

**Problemas do GARCH com parametros fixos:**

1. **Persistencia excessiva**: $\alpha + \beta \approx 1$ (IGARCH), porque o modelo tenta
   acomodar tanto periodos calmos quanto turbulentos com os mesmos parametros
2. **Resposta simetrica a choques**: a mesma dinâmica em crises e em periodos normais
3. **Previsoes de longo prazo ruins**: convergencia lenta para o nivel incondicional

O **MS-GARCH** resolve isso permitindo parametros **diferentes** em cada regime:

$$\sigma_t^2(S_t) = \omega_{S_t} + \alpha_{S_t} \epsilon_{t-1}^2 + \beta_{S_t} h_{t-1}$$

onde $h_{t-1}$ e a variancia "colapsada" (Gray, 1996):

$$h_{t-1} = \sum_{j=1}^{K} P(S_{t-1} = j \mid \mathcal{Y}_{t-1}) \cdot \sigma^2_{t-1}(j)$$

Isso permite que:
- **Regime calmo**: $\omega_1$ pequeno, $\alpha_1 + \beta_1$ alta (alta persistencia dentro do regime)
- **Regime turbulento**: $\omega_2$ grande, $\alpha_2$ alta (forte reacao a choques)

In [ ]:
# Estimar GARCH(1,1) padrao como baseline
model_garch = GARCH(returns.values, p=1, q=1)
results_garch = model_garch.fit()
print(results_garch.summary())

# Observar a persistencia
persistence_garch = results_garch.persistence()
print(f'\nPersistencia (alpha + beta): {persistence_garch:.4f}')
print(f'Meia-vida: {results_garch.half_life():.1f} dias')

if persistence_garch > 0.95:
    print('\n** ATENCAO: Persistencia muito alta (> 0.95)!')
    print('   Isso pode indicar que o modelo nao captura mudancas de regime.')

# Plotar retornos e volatilidade condicional
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(returns.index, returns.values, color='black', lw=0.5)
axes[0].set_title('Retornos S&P 500', fontsize=12)
axes[0].set_ylabel('Retorno')
axes[0].grid(True, alpha=0.3)

axes[1].plot(returns.index, results_garch.conditional_volatility,
             color='darkred', lw=0.8)
axes[1].set_title(f'Volatilidade Condicional - GARCH(1,1) (persistencia={persistence_garch:.4f})', fontsize=12)
axes[1].set_ylabel('Volatilidade')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. MS(2)-GARCH(1,1)

O modelo **MS(2)-GARCH(1,1)** especifica dois conjuntos de parametros GARCH:

**Regime 1 (calmo):**
$$\sigma_t^2(1) = \omega_1 + \alpha_1 \epsilon_{t-1}^2 + \beta_1 h_{t-1}$$

**Regime 2 (turbulento):**
$$\sigma_t^2(2) = \omega_2 + \alpha_2 \epsilon_{t-1}^2 + \beta_2 h_{t-1}$$

### Abordagem de Gray (1996)

Para manter o modelo tratavel, Gray (1996) propoe "colapsar" a variancia condicional:

$$h_{t-1} = \sum_{j=1}^{K} P(S_{t-1} = j \mid \mathcal{Y}_{t-1}) \left[ \sigma^2_{t-1}(j) + (\mu_{t-1}(j) - \bar{\mu}_{t-1})^2 \right]$$

Isso evita que o numero de caminhos cresça exponencialmente com $T$.

### Restricoes de estacionariedade

Para cada regime: $\alpha_s + \beta_s < 1$ e $\omega_s > 0$, $\alpha_s \geq 0$, $\beta_s \geq 0$.

In [ ]:
# Estimar MS(2)-GARCH(1,1) com archbox
model_ms = MarkovSwitchingGARCH(
    endog=returns.values,
    k_regimes=2,
    p=1,
    q=1,
    method='gray',
)
results_ms = model_ms.fit(method='em', maxiter=500, verbose=True)

print('\n' + results_ms.summary())
print(f'\nConvergiu: {results_ms.converged}')
print(f'Iteracoes: {results_ms.n_iter}')
print(f'Log-likelihood: {results_ms.loglike:.4f}')
print(f'AIC: {results_ms.aic:.4f}')
print(f'BIC: {results_ms.bic:.4f}')

## 3. Regime de alta vs baixa volatilidade

Comparando os parametros estimados entre os 2 regimes, podemos entender as diferencas
na dinamica de volatilidade:

| Parametro | Regime 1 (calmo) | Regime 2 (turbulento) |
|-----------|------------------|-----------------------|
| $\omega$ | Baixo | Alto |
| $\alpha$ | Baixo | Alto |
| $\beta$ | Alto | Moderado |
| $\alpha + \beta$ | Alta persistencia | Menor persistencia |
| Vol. incondicional | Baixa | Alta |

A **variancia incondicional** de cada regime e:

$$\bar{\sigma}^2_s = \frac{\omega_s}{1 - \alpha_s - \beta_s}$$

Tipicamente, $\bar{\sigma}^2_2 \gg \bar{\sigma}^2_1$.

In [ ]:
# Comparar parametros entre os 2 regimes
print('='*60)
print('Parametros GARCH por Regime')
print('='*60)

regime_data = []
for regime_id, params in results_ms.regime_params.items():
    omega = params.get('omega', 0)
    alpha = params.get('alpha', 0)
    beta = params.get('beta', 0)
    persistence = params.get('persistence', alpha + beta)
    uncond_var = omega / (1 - persistence) if persistence < 1 else float('inf')
    uncond_vol = np.sqrt(uncond_var) if uncond_var < float('inf') else float('inf')

    print(f'\n--- Regime {regime_id} ---')
    print(f'  omega:  {omega:.8f}')
    print(f'  alpha:  {alpha:.4f}')
    print(f'  beta:   {beta:.4f}')
    print(f'  Persistencia (alpha+beta): {persistence:.4f}')
    print(f'  Vol. incondicional (anualizada): {uncond_vol * np.sqrt(252):.4f}')

    regime_data.append({
        'Regime': regime_id, 'omega': omega, 'alpha': alpha, 'beta': beta,
        'Persistencia': persistence, 'Vol. Incond.': uncond_vol
    })

# Tabela comparativa
df_params = pd.DataFrame(regime_data).set_index('Regime')
print('\n\nTabela comparativa:')
print(df_params.to_string())

# Matriz de transicao
P = results_ms.transition_matrix
print("\n\nMatriz de transicao:")
print(f"  Linhas somam 1: {[f'{P[i].sum():.6f}' for i in range(P.shape[0])]}")

fig = plot_transition_matrix(
    P,
    regime_labels=['Calmo', 'Turbulento'],
    title='Matriz de Transicao - MS-GARCH',
)
plt.show()

# Duracoes esperadas
duracoes = results_ms.expected_durations()
print(f'\nDuracao esperada regime calmo: {duracoes[0]:.1f} dias ({duracoes[0]/252:.1f} anos)')
print(f'Duracao esperada regime turbulento: {duracoes[1]:.1f} dias ({duracoes[1]/252:.1f} anos)')

# Probabilidades ergoticas
ergodic = results_ms.ergodic_probabilities()
print('\nProbabilidades ergoticas:')
print(f'  Regime calmo: {ergodic[0]:.1%}')
print(f'  Regime turbulento: {ergodic[1]:.1%}')

## 4. Volatilidade condicional com regime-switching

A volatilidade condicional do MS-GARCH e uma **media ponderada** das volatilidades
de cada regime, ponderada pelas probabilidades filtradas:

$$\sigma_t^2 = \sum_{s=1}^{K} P(S_t = s \mid \mathcal{Y}_t) \cdot \sigma_t^2(s)$$

Essa volatilidade agregada:
- **Sobe rapidamente** quando o regime turbulento se torna mais provavel
- **Cai rapidamente** quando o regime calmo volta a dominar
- E mais **flexivel** que o GARCH padrao, pois nao depende apenas de $\alpha + \beta$

Visualizar a volatilidade **colorida por regime** ajuda a identificar os periodos
de turbulencia no mercado.

In [ ]:
# Classificar regimes
regimes = results_ms.classify(threshold=0.5)

# Calcular volatilidade condicional ponderada por regime
# sigma2_t = sum_s P(S_t=s|Y_t) * sigma2_t(s)
# Usar _sigma2 do modelo se disponivel, caso contrario usar smoothed_probs
if model_ms._sigma2 is not None:
    vol_weighted = np.sqrt(np.sum(results_ms.smoothed_probs * model_ms._sigma2, axis=1))
else:
    # Fallback: usar variancia dos retornos ponderada por regime
    vol_weighted = np.abs(returns.values)  # proxy

# Plotar volatilidade colorida por regime
fig = plot_ms_volatility(
    dates=returns.index,
    returns=returns.values,
    volatility=vol_weighted,
    regimes=regimes,
    title='MS(2)-GARCH(1,1) Volatilidade com Regimes',
)
plt.show()

# Plotar probabilidades de regime suavizadas
fig = plot_regime_probabilities(
    dates=returns.index,
    series=returns.values,
    probabilities=results_ms.smoothed_probs,
    regime_labels=['Calmo', 'Turbulento'],
    title='MS-GARCH: Probabilidades de Regime',
)
plt.show()

# Verificar que probabilidades estao entre 0 e 1
print("Probabilidades suavizadas:")
print(f"  Min: {results_ms.smoothed_probs.min():.6f}")
print(f"  Max: {results_ms.smoothed_probs.max():.6f}")
print(f"  Soma por linha (media): {results_ms.smoothed_probs.sum(axis=1).mean():.6f}")

# Proporcao do tempo em cada regime
for r in np.unique(regimes):
    n_r = np.sum(regimes == r)
    print(f"  Regime {r}: {n_r} dias ({n_r/len(regimes):.1%})")

## 5. MS-GARCH vs GARCH padrao

Comparamos os dois modelos em termos de:

1. **Criterios de informacao**: AIC e BIC
2. **Qualidade de ajuste**: volatilidade estimada vs realizada
3. **Diagnosticos**: autocorrelacao dos residuos padronizados

O MS-GARCH deve apresentar:
- **Menor AIC/BIC** se os regimes forem reais (nao apenas ruido)
- **Menor persistencia** dentro de cada regime (vs persistencia inflada do GARCH)
- **Melhor captura** de mudancas abruptas na volatilidade

In [ ]:
# Comparacao quantitativa GARCH(1,1) vs MS(2)-GARCH(1,1)
print('='*60)
print('Comparacao GARCH(1,1) vs MS(2)-GARCH(1,1)')
print('='*60)
print(f'  GARCH:    AIC={results_garch.aic:.2f}, BIC={results_garch.bic:.2f}')
print(f'  MS-GARCH: AIC={results_ms.aic:.2f}, BIC={results_ms.bic:.2f}')
print(f'  Log-lik GARCH:    {results_garch.loglike:.2f}')
print(f'  Log-lik MS-GARCH: {results_ms.loglike:.2f}')
print(f'  N params GARCH:    {len(results_garch.params)}')
print(f'  N params MS-GARCH: {results_ms.n_params}')

# Comparar persistencia
print('\nPersistencia:')
print(f'  GARCH padrao: {persistence_garch:.4f}')
for regime_id, params in results_ms.regime_params.items():
    alpha = params.get('alpha', 0)
    beta = params.get('beta', 0)
    pers = params.get('persistence', alpha + beta)
    print(f'  MS-GARCH Regime {regime_id}: {pers:.4f}')

# Resumo
melhor_aic = 'GARCH' if results_garch.aic < results_ms.aic else 'MS-GARCH'
melhor_bic = 'GARCH' if results_garch.bic < results_ms.bic else 'MS-GARCH'
print(f'\nModelo preferido por AIC: {melhor_aic}')
print(f'Modelo preferido por BIC: {melhor_bic}')

# Grafico comparativo de barras
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# AIC/BIC
models = ['GARCH(1,1)', 'MS-GARCH']
aic_vals = [results_garch.aic, results_ms.aic]
bic_vals = [results_garch.bic, results_ms.bic]

x = np.arange(len(models))
width = 0.35
axes[0].bar(x - width/2, aic_vals, width, label='AIC', color='steelblue')
axes[0].bar(x + width/2, bic_vals, width, label='BIC', color='coral')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].set_title('Criterios de Informacao')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Log-likelihood
axes[1].bar(models, [results_garch.loglike, results_ms.loglike], color=['steelblue', 'coral'])
axes[1].set_title('Log-Likelihood')
axes[1].grid(True, alpha=0.3, axis='y')

# Persistencia
pers_labels = ['GARCH']
pers_vals = [persistence_garch]
for regime_id, params in results_ms.regime_params.items():
    pers_labels.append(f'MS Reg.{regime_id}')
    pers_vals.append(params.get('persistence', params.get('alpha', 0) + params.get('beta', 0)))

axes[2].bar(pers_labels, pers_vals, color=['steelblue', '#2ecc71', '#e74c3c'])
axes[2].axhline(y=1.0, color='black', linestyle='--', alpha=0.3, label='IGARCH')
axes[2].set_title('Persistencia por Modelo/Regime')
axes[2].set_ylim(0, 1.1)
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\nInterpretacao:')
print('O MS-GARCH resolve o problema de persistencia excessiva!')
print('Dentro de cada regime, alpha+beta e menor que no GARCH padrao.')
print('A "alta persistencia" do GARCH era um artefato de misturar regimes.')

## Conclusao

Neste notebook, aprendemos:

- Por que o **GARCH padrao** pode apresentar **persistencia excessiva** quando ha mudanca de regime
- Como o **MS(2)-GARCH(1,1)** modela volatilidade com regimes calmo e turbulento
- Como interpretar os **parametros por regime** (volatilidade incondicional, persistencia)
- Como visualizar a **volatilidade condicional colorida por regime**
- Como comparar MS-GARCH vs GARCH padrao usando **AIC/BIC** e diagnosticos

O MS-GARCH e particularmente util quando:
- A volatilidade apresenta **saltos abruptos** (crises financeiras)
- O GARCH padrao mostra **persistencia proxima de 1** (IGARCH)
- Voce precisa de estimativas de **probabilidade de estar em crise**

### Referencias

- Haas, M., Mittnik, S. & Paolella, M.S. (2004). A New Approach to Markov-Switching GARCH Models. *Journal of Financial Econometrics*, 2(4), 493-530.
- Gray, S.F. (1996). Modeling the Conditional Distribution of Interest Rates as a Regime-Switching Process. *Journal of Financial Economics*, 42, 27-62.
- Hamilton, J.D. (1989). A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle. *Econometrica*, 57(2), 357-384.
- Krolzig, H.-M. (1997). *Markov-Switching Vector Autoregressions*. Springer.